In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools.retriever import create_retriever_tool
from langchain_classic.agents import AgentExecutor, ZeroShotAgent, create_openai_tools_agent
from langchain_community.document_loaders import ArxivLoader, WebBaseLoader
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.embeddings import OpenAIEmbeddings, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama.llms import OllamaLLM
from langchain_classic import hub

In [2]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)

wikipedia_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [3]:
wikipedia_tool.name

'wikipedia'

In [5]:
arxiv_loader = ArxivLoader(query="cat:cs.AI", doc_content_chars_max=5)
arxiv_documents = arxiv_loader.load()

print(f"All arXiv documents: {arxiv_documents}\n")

All arXiv documents: []



In [6]:
web_loader = WebBaseLoader("https://fastapi.tiangolo.com/advanced/events/")
web_documents = web_loader.load()

print(f"All web documents: {web_documents}\n")

All web documents: [Document(metadata={'source': 'https://fastapi.tiangolo.com/advanced/events/', 'title': 'Lifespan Events - FastAPI', 'description': 'FastAPI framework, high performance, easy to learn, fast to code, ready for production', 'language': 'en'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLifespan Events - FastAPI\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n          Skip to content\n        \n\n\n\n\n\n\n\n\n\n\n Join the FastAPI Cloud waiting list 🚀\n      \n\n\n\n\n\n Follow @fastapi on X (Twitter) to stay updated\n      \n\n\n\n\n\n Follow FastAPI on LinkedIn to stay updated\n      \n\n\n\n\n\n Subscribe to the FastAPI and friends newsletter 🎉\n      \n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\nsponsor\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n            FastAPI\n          \n\n\n\n  

In [ ]:
docs = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0).split_documents(web_documents)
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
chroma_db = Chroma.from_documents(docs, embeddings)

In [ ]:
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(web_documents)

vectorstore = FAISS.from_documents(documents, OllamaEmbeddings(model="llama2"))

retriever = vectorstore.as_retriever()
retriever

In [ ]:
retriever.get_relevant_documents("What are FastAPI events?")

In [ ]:
## The retriever is able to find the relevant information about FastAPI events from the web documents we loaded and split into chunks.
## Create a retriever tool and use it to create an agent that can answer questions about FastAPI events. create_retriever_tool() method from langchain_community.tools can be used to create a retriever tool from the retriever we just created. Then, we can use this tool to create an agent that can answer questions about FastAPI events.
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="FastAPI_Events_Search",
    description="Useful for answering questions about FastAPI events. For any question about FastAPI events, use this tool to search for relevant information in the web documents."
)

In [ ]:
retriever_tool.name

In [ ]:
## ARXIV Tool
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=5)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [ ]:
tools = [wikipedia_tool, arxiv, retriever_tool]

In [21]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["OLLAMA_API_URL"] = "http://localhost:11434"

In [ ]:
ollama = OllamaLLM(model="llama2", temperature=0.5)

In [ ]:
prompt = hub.pull("fastapi_events_agent_prompt")
prompt.messages

In [ ]:
agent = create_openai_tools_agent(llm=ollama, tools=tools, prompt=prompt)
# agent = ZeroShotAgent(llm=ollama, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
# Retriever tool is going to be used
agent_executor.invoke({"input": "What are FastAPI events? Tell me about them."})

In [ ]:
# Wikipedia tool is going to be used
agent_executor.invoke({"input": "What is Machine Learning?"})

In [ ]:
# ARXIV tool is going to be used
agent_executor.invoke({"input": "What is the paper 'Attention Is All You Need' about?"})